# **Simple Preprocess Baseline**

-----
## feature importance observations (baseline models)

### saw:
- across logistic regression, random forest, and xgb:
  - top features are consistent:
    - ICU type (unittype_*)
    - disease history flags (hx_*)
    - severity scores (aps, apachescore)
    - age and core vitals/labs
- ABG (arterial blood gas) variables (ph, fio2, pco2, pao2) do NOT appear as strong predictors

### thought process:
- even w/ more complex models (RF, XGB), ABG features did not gain importance
- confirms:
  - high missingness (~84%) limits usefulness
  - imputation likely diluted signal
- so far the models relies more on:
  - baseline patient condition
  - icu context
  - early vitals/labs

### key takeaway
- raw ABG values are not useful in current form
- presence of ABG (whether it was measured) may still contain signal

---

## what we can do next

### make a final decision for baseline
- drop: ABG variables 

### feature engineering 
- add:
  - is_abg_measured (binary flag)

### next goals...
- rebuild model with:
  - cleaned dataset
  - engineered features
- now focus on improving recall and f1


-------

In [50]:
# imports

import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [4]:
# load in data
df = pd.read_csv("simple_preprocess.csv")
print("shape:", df.shape)
df.head()

shape: (2520, 77)


,gender,age,ethnicity,admissionheight,admissionweight,unittype,bmi,dialysis,wbc,respiratoryrate,...,sbp_min,dbp_min,map_min,vitals_missing,hr_range,temp_range,numbedscategory,teachingstatus,region,bad_outcome
0,Female,87.0,Caucasian,157.5,NaN,Med-Surg ICU,NaN,NaN,NaN,NaN,...,84.0,44.0,58.0,1,48.0,1.8,<100,f,Midwest,1
1,Female,87.0,Caucasian,157.5,46.5,Med-Surg ICU,18.745276,0.0,10.2,39.0,...,84.0,44.0,58.0,1,44.0,1.8,<100,f,Midwest,0
2,Male,76.0,Caucasian,167.0,77.5,SICU,27.788734,0.0,11.7,60.0,...,53.0,36.0,47.0,1,15.0,1.8,<100,f,Midwest,0
3,Female,34.0,Caucasian,172.7,60.3,Med-Surg ICU,20.217741,0.0,7.9,6.0,...,84.0,44.0,58.0,1,50.0,1.8,<100,f,Midwest,0
4,Male,61.0,Caucasian,177.8,91.7,SICU,29.007201,0.0,21.1,41.0,...,84.0,44.0,58.0,1,36.0,1.8,<100,f,Midwest,0


In [6]:
# target variable
target_col = "bad_outcome"

# separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (2520, 76)
Target shape: (2520,)


------
### **train/test split**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y 
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (2016, 76)
Test shape: (504, 76)


In [11]:
# col types

# num cols
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# cat cols
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:", len(num_cols))
print("categorical columns:", len(cat_cols))

numeric columns: 70
categorical columns: 6


In [13]:
# preprocessing

# num
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# cat
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# combine both
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

--------
### **same baseline model**

In [18]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [20]:
# train model

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'admissionheight',
                                                   'admissionweight', 'bmi',
                                                   'dialysis', 'wbc',
                                                   'respiratoryrate', 'sodium',
                                                   'heartrate', 'meanbp', 'ph',
                                                   'hematocrit', 'creatinine',
                                                   'albumin', 'pao2', 'pco2',
                                                   'bun', 'glucose',
                                                   'b...
                                                   'apache_missing', 'aids',
                                                   'hepaticfailure', 'lymphoma',
                                                   'metastaticcancer',
                                                   'leukemia',
                                                   'immunosuppression', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'ethnicity',
                                                   'unittype',
                                                   'numbedscategory',
                                                   'teachingstatus',
                                                   'region'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [24]:
# evaluation

# predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# metrics
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC AUC:", roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[366  22]
 [ 63  53]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.94      0.90       388
           1       0.71      0.46      0.55       116

    accuracy                           0.83       504
   macro avg       0.78      0.70      0.73       504
weighted avg       0.82      0.83      0.82       504

ROC AUC: 0.8197209384998222


------
### **feature importance**

In [27]:
feature_names = model.named_steps['preprocessing'].get_feature_names_out()

# get coefficients
coefficients = model.named_steps['classifier'].coef_[0]

# create dataframe
importance_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefficients,
    "abs_coef": np.abs(coefficients)
})

# sort by importance
importance_df = importance_df.sort_values(by="abs_coef", ascending=False)

# show top 50
importance_df.head(50)

,feature,coef,abs_coef
48,num__hx_heme,-0.629147,0.629147
84,cat__unittype_Neuro ICU,0.578609,0.578609
88,cat__numbedscategory_<100,-0.546874,0.546874
49,num__hx_none,-0.496839,0.496839
93,cat__region_Northeast,-0.443681,0.443681
82,cat__unittype_MICU,-0.424796,0.424796
80,cat__unittype_CTICU,0.384780,0.384780
76,cat__ethnicity_Native American,-0.363735,0.363735
81,cat__unittype_Cardiac ICU,-0.344030,0.344030
20,num__aps_missing,0.325475,0.325475


In [29]:
# filter for ABG-related features, looking as missingness was ~83% earlier
importance_df[importance_df["feature"].str.contains("ph|fio2|pco2|pao2", case=False)]

,feature,coef,abs_coef
21,num__acutephysiologyscore,0.163428,0.163428
14,num__pao2,-0.145532,0.145532
19,num__fio2,0.123027,0.123027
10,num__ph,-0.078824,0.078824
15,num__pco2,-0.024051,0.024051
26,num__lymphoma,0.008545,0.008545


In [31]:
# check processed feature shape
X_train_processed = model.named_steps['preprocessing'].transform(X_train)

print("Processed train shape:", X_train_processed.shape)

Processed train shape: (2016, 96)


---------------
### **drop high missingness cols and running log reg again to try**

In [34]:
df_no_abg = df.copy()

In [36]:
abg_cols = ["ph", "fio2", "pco2", "pao2"]

df_no_abg = df_no_abg.drop(columns=abg_cols, errors="ignore")

print("new shape:", df_no_abg.shape)

new shape: (2520, 73)


In [38]:
X_no_abg = df_no_abg.drop(columns=["bad_outcome"])
y_no_abg = df_no_abg["bad_outcome"]

In [41]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_no_abg, y_no_abg,
    test_size=0.2,
    random_state=42,
    stratify=y_no_abg
)

In [43]:
num_cols2 = X_train2.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols2 = X_train2.select_dtypes(include=["object"]).columns.tolist()

In [45]:
model_no_abg = Pipeline([
    ("preprocessing", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols2),
        
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols2)
    ])),
    
    ("classifier", LogisticRegression(max_iter=1000))
])

In [47]:
model_no_abg.fit(X_train2, y_train2)

y_pred2 = model_no_abg.predict(X_test2)
y_prob2 = model_no_abg.predict_proba(X_test2)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test2, y_pred2))

print("\nClassification Report:")
print(classification_report(y_test2, y_pred2))

print("ROC AUC:", roc_auc_score(y_test2, y_prob2))

Confusion Matrix:
[[368  20]
 [ 61  55]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.95      0.90       388
           1       0.73      0.47      0.58       116

    accuracy                           0.84       504
   macro avg       0.80      0.71      0.74       504
weighted avg       0.83      0.84      0.83       504

ROC AUC: 0.8183656238890864


-------
### **re-running pipeline once more with tuning**

In [52]:
# function to train & evaluate

# runs one model on one df
# andles split, preprocessing, training, and evaluation
def run_model_experiment(df_input, model_name, classifier):
    
    # separate features and target
    X = df_input.drop(columns=["bad_outcome"])
    y = df_input["bad_outcome"]
    
    # split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # detect column types
    num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    
    # num preprocessing
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    # cat
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    # combine 
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])
    
    # full pipeline
    model = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", classifier)
    ])
    
    # fit model
    model.fit(X_train, y_train)
    
    # predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # metrics
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_prob)
    
    # save key results
    results = {
        "model": model_name,
        "precision_1": report["1"]["precision"],
        "recall_1": report["1"]["recall"],
        "f1_1": report["1"]["f1-score"],
        "roc_auc": auc,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1]
    }
    
    return results

In [54]:
# make both datasets once again

# original cleaned dataframe (with ABG)
df_with_abg = df.copy()

# version without ABG
abg_cols = ["ph", "fio2", "pco2", "pao2"]
df_without_abg = df.copy().drop(columns=abg_cols, errors="ignore")

print("With ABG shape:", df_with_abg.shape)
print("Without ABG shape:", df_without_abg.shape)

With ABG shape: (2520, 77)
Without ABG shape: (2520, 73)


In [56]:
# models, logreg, rf, xgb

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

In [58]:
# run all experiments

all_results = []

for model_name, clf in models.items():
    # run on data with ABG
    result_with = run_model_experiment(df_with_abg, model_name + " (with ABG)", clf)
    all_results.append(result_with)
    
    # run on data without ABG
    result_without = run_model_experiment(df_without_abg, model_name + " (without ABG)", clf)
    all_results.append(result_without)

In [60]:
# results table

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by=["recall_1", "f1_1", "roc_auc"],
    ascending=False
)

results_df

,model,precision_1,recall_1,f1_1,roc_auc,tn,fp,fn,tp
3,Random Forest (without ABG),0.688889,0.534483,0.601942,0.862180,360,28,54,62
5,XGBoost (without ABG),0.794521,0.500000,0.613757,0.854915,373,15,58,58
4,XGBoost (with ABG),0.783784,0.500000,0.610526,0.852959,372,16,58,58
2,Random Forest (with ABG),0.698795,0.500000,0.582915,0.865602,363,25,58,58
1,Logistic Regression (without ABG),0.733333,0.474138,0.575916,0.818366,368,20,61,55
0,Logistic Regression (with ABG),0.706667,0.456897,0.554974,0.819721,366,22,63,53


-------
### looking at feature importance again

In [65]:
# log reg
importance_logreg = importance_df.copy()

importance_logreg.head(15)

,feature,coef,abs_coef
48,num__hx_heme,-0.629147,0.629147
84,cat__unittype_Neuro ICU,0.578609,0.578609
88,cat__numbedscategory_<100,-0.546874,0.546874
49,num__hx_none,-0.496839,0.496839
93,cat__region_Northeast,-0.443681,0.443681
82,cat__unittype_MICU,-0.424796,0.424796
80,cat__unittype_CTICU,0.384780,0.384780
76,cat__ethnicity_Native American,-0.363735,0.363735
81,cat__unittype_Cardiac ICU,-0.344030,0.344030
20,num__aps_missing,0.325475,0.325475


In [67]:
# random forest feature importance

# get feature names
rf_model = models["Random Forest"]
rf_pipeline = run_model_experiment  

# refit to get pipeline
rf_full = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ))
])

rf_full.fit(X_train, y_train)

# get feature names
feature_names = rf_full.named_steps['preprocessing'].get_feature_names_out()

# get importances
rf_importances = rf_full.named_steps['classifier'].feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importances
}).sort_values(by="importance", ascending=False)

rf_importance_df.head(15)

,feature,importance
33,num__pred_missing,0.069209
20,num__aps_missing,0.065606
22,num__apachescore,0.053735
34,num__total_diagnoses,0.043632
21,num__acutephysiologyscore,0.042706
16,num__bun,0.039495
6,num__respiratoryrate,0.035405
9,num__meanbp,0.031752
23,num__apache_missing,0.029803
8,num__heartrate,0.028682


In [69]:
# xgb importance

xgb_full = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_full.fit(X_train, y_train)

# feature names
feature_names = xgb_full.named_steps['preprocessing'].get_feature_names_out()

# importance
xgb_importances = xgb_full.named_steps['classifier'].feature_importances_

xgb_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_importances
}).sort_values(by="importance", ascending=False)

xgb_importance_df.head(15)

,feature,importance
20,num__aps_missing,0.148045
33,num__pred_missing,0.135015
23,num__apache_missing,0.028515
22,num__apachescore,0.018789
48,num__hx_heme,0.016951
21,num__acutephysiologyscore,0.014262
16,num__bun,0.013904
63,num__sao2_min,0.013695
56,num__non_drug_allergy,0.013664
81,cat__unittype_Cardiac ICU,0.013385


------------